# SESANS and SEMSANS: fitting in real space

Spin-echo SANS (SESANS) measures the depolarisation of a neutron beam against
the **spin-echo length** ξ, a real-space distance. IQuana fits it with the same
I(Q) models as SANS data: sasmodels turns each model into the SESANS quantity,
ln(P/P₀)/(tλ²), through a Hankel transform. Reduced SEMSANS data (spin-echo
modulated SANS) is the same measurement made another way, with the visibility
ratio Vs/V₀ in place of P/P₀.

1. [A .ses file](#1.-A-.ses-file)
2. [The fit range](#2.-The-fit-range)
3. [Time of flight and a structure factor](#3.-Time-of-flight-and-a-structure-factor)
4. [Reduced SEMSANS data from arrays](#4.-Reduced-SEMSANS-data-from-arrays)
5. [Structure smaller than the first step](#5.-Structure-smaller-than-the-first-step)
6. [What is refused, and why](#6.-What-is-refused,-and-why)
7. [Save and reopen](#7.-Save-and-reopen)

The data are the two SESANS examples that ship with sasdata. See
`docs/sesans.md` for the background, including which acceptance angle to give.

In [ ]:
import os
import warnings

import numpy as np
import sasdata

from iquana import IQuana, data_ops

SES_DIR = os.path.join(os.path.dirname(sasdata.__file__), 'example_data', 'sesans_data')
DELFT = os.path.join(SES_DIR, 'sphere2micron.ses')  # 2 um polystyrene spheres, one wavelength
ISIS = os.path.join(SES_DIR, 'sphere_isis.ses')  # PMMA hard spheres, time of flight


def sphere_setup(fitter, radius):
    """A dilute sphere with unit contrast: radius and scale (phi * drho^2) free."""
    fitter.set_model('sphere')
    fitter.set_param('radius', value=radius, min=radius / 5, max=radius * 5, vary=True)
    fitter.set_param('scale', value=0.01, min=1e-4, max=1, vary=True)
    fitter.set_param('sld', value=1.0, vary=False)
    fitter.set_param('sld_solvent', value=0.0, vary=False)

## 1. A .ses file

`load_data()` reads the file and recognises it as SESANS data. The summary
names what the transform uses: the wavelength and the acceptance angle.

In [ ]:
fitter = IQuana()
fitter.load_data(DELFT)
fitter.is_sesans

In [ ]:
sphere_setup(fitter, radius=9000)
result = fitter.fit()

The fit recovers the 2 µm spheres. The plot has linear axes: the
depolarisation starts at zero and falls to a negative plateau at the sphere
diameter, which a log axis could not show.

In [ ]:
fitter.plot_results()

## 2. The fit range

`set_q_range()` takes spin-echo lengths in Å for SESANS data. The range
selects the fitted points; the transform is still computed on the whole grid,
so the curve at a kept point does not change.

In [ ]:
fitter.set_q_range(5000, 30000)
ranged = fitter.fit()
ranged['n_points']

In [ ]:
fitter.plot_results()

In [ ]:
fitter.reset_q_range()
result = fitter.fit()

## 3. Time of flight and a structure factor

The ISIS file has one wavelength per point. Models, structure factors and
polydispersity work as for SANS: they act on I(Q) before the transform.

In [ ]:
tof = IQuana()
tof.load_data(ISIS)
sphere_setup(tof, radius=1000)
tof.set_structure_factor('hardsphere')
tof.set_param('scale', value=0.3, min=0.01, max=0.7, vary=True)
tof_result = tof.fit()
tof.plot_results()

## 4. Reduced SEMSANS data from arrays

Reduced SEMSANS data is usually a table of spin-echo length, visibility ratio
and its error: read it with `np.loadtxt` and give it to `data_ops.sesans()`.
With `polarisation=` the ratio is turned into ln(P/P₀)/(tλ²), which needs the
sample thickness. Here the ratio is made from the Delft data, so the fit should
agree with section 1.

In [ ]:
loaded = data_ops.load(DELFT)
thickness, lam = 0.2, 2.11
ratio = np.exp(thickness * lam**2 * loaded.y)
ratio_error = ratio * thickness * lam**2 * loaded.dy

semsans = data_ops.sesans(
    loaded.x,                 # spin-echo length, A
    polarisation=ratio,       # P/P0, or Vs/V0 for SEMSANS
    error=ratio_error,        # one standard deviation of the ratio
    wavelength=lam,           # A: one number, or one per point
    thickness=thickness,      # cm
    acceptance=0.0168,        # rad: see docs/sesans.md for SEMSANS
)
arrays = IQuana()
arrays.set_data(semsans)
sphere_setup(arrays, radius=9000)
array_result = arrays.fit()
array_result['parameters']['radius']['value'], result['parameters']['radius']['value']

## 5. Structure smaller than the first step

sasmodels takes the largest q of its transform from the first step between
spin-echo lengths: 1172 Å on the Delft grid. Structure much smaller than that
is cut off. After every SESANS fit IQuana computes the curve once more with a
finer limit and warns, without stopping, when the curve moves. Here the data
are 1000 Å spheres, the model itself with 5 % error bars.

In [ ]:
model = IQuana()
model.set_data(semsans)
sphere_setup(model, radius=1000)
model.set_param('scale', value=0.13)
curve = model.calculate()

small = IQuana()
small.set_data(
    data_ops.sesans(
        loaded.x,
        depolarisation=curve,
        error=np.full_like(curve, 0.05 * np.abs(curve).max()),
        wavelength=lam,
        acceptance=0.0168,
    )
)
sphere_setup(small, radius=1200)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    small.fit()
[str(w.message) for w in caught if 'integration limit' in str(w.message)]

A silent check is not proof: it compares two numerical limits, so scattering
beyond both goes unnoticed.

## 6. What is refused, and why

Settings that mean nothing for SESANS data are refused with a message that
says what to do instead.

In [ ]:
for attempt in (
    lambda: fitter.set_resolution('pinhole', dq_over_q=0.1),
    lambda: fitter.set_weighting('sqrt'),
    lambda: fitter.plot_results(log_scale=True),
    lambda: fitter.calculate(q=np.linspace(1e-3, 1e-2, 5)),
):
    try:
        attempt()
    except ValueError as error:
        print(f'{error}\n')

## 7. Save and reopen

A SESANS analysis is saved as schema version 4, with its fit result.

In [ ]:
fitter.save_analysis('polystyrene_sesans.json')
reopened = IQuana.load_analysis('polystyrene_sesans.json')
reopened.fit_result is not None

In [ ]:
os.remove('polystyrene_sesans.json')